In [2]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.appName('weather').getOrCreate()
df = spark.read.csv('weather_forecast_data.csv', header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- Cloud_Cover: double (nullable = true)
 |-- Pressure: double (nullable = true)
 |-- Rain: string (nullable = true)

+------------------+-----------------+------------------+------------------+------------------+-------+
|       Temperature|         Humidity|        Wind_Speed|       Cloud_Cover|          Pressure|   Rain|
+------------------+-----------------+------------------+------------------+------------------+-------+
|23.720337598183118|89.59264065174611| 7.335604391040214|50.501693832913155| 1032.378758690279|   rain|
|27.879734159310487|46.48970403534824| 5.952483593282764| 4.990052927536981| 992.6141895121403|no rain|
|25.069084401791095|83.07284289257146|1.3719918180799207|14.855783939243427|1007.2316201172738|no rain|
|23.622079574922424|74.36775792086564| 7.050550632784658| 67.25528206034686| 982.6320127095369|   rain|
|20.591369983472617|96

## Ekplorasi data

In [6]:
from pyspark.sql.functions import *

print("Jumlah baris:", df.count())
print("Jumlah kolom:", len(df.columns))

# Statistik numerik
df.describe().show()

# Cek missing values
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()


Jumlah baris: 2500
Jumlah kolom: 6
+-------+------------------+------------------+--------------------+--------------------+------------------+-------+
|summary|       Temperature|          Humidity|          Wind_Speed|         Cloud_Cover|          Pressure|   Rain|
+-------+------------------+------------------+--------------------+--------------------+------------------+-------+
|  count|              2500|              2500|                2500|                2500|              2500|   2500|
|   mean|22.581725199399855|  64.3470944576354|   9.906254759749315|   49.65810425506773|1014.3123358612227|   NULL|
| stddev| 7.326996316570904|19.954739172195946|   5.780316036232053|  29.123103660104825|20.196432793986787|   NULL|
|    min|10.001842485827652|30.005071474694454|0.009819276000868626|0.015038108085552171| 980.0144863937445|no rain|
|    max| 34.99521445292413| 99.99748129798218|    19.9991322551545|   99.99779517807228|1049.9855933650479|   rain|
+-------+------------------+-

## Data prapocecing

In [7]:
# Isi nilai kosong
from pyspark.ml.feature import StringIndexer

# Isi kategori kosong
string_cols = [c for c, t in df.dtypes if t == 'string']
df = df.fillna('missing', subset=string_cols)

# Isi numerik dengan median
num_cols = [c for c, t in df.dtypes if t != 'string']

for c in num_cols:
    median_val = df.approxQuantile(c, [0.5], 0.1)[0]
    df = df.fillna({c: median_val})


In [8]:
label_indexer = StringIndexer(
    inputCol="Rain",
    outputCol="label"
)


In [9]:
categorical_cols = [c for c, t in df.dtypes if t == 'string' and c != 'Rain']
indexers = [
    StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid='keep')
    for c in categorical_cols
]


## Feature Vector + Normalisasi

In [10]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

feature_cols = [c for c in df.columns if c not in ['Rain']]  # selain label
feature_cols = [c for c in feature_cols if c not in categorical_cols]  # buang kolom string

# tambahkan kolom idx hasil encoding
feature_cols = feature_cols + [c+"_idx" for c in categorical_cols]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features")


## Train–Test Split

In [11]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)


## Model Logistic Regression

In [12]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")


## Pipeline Full

In [13]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=indexers + [
    label_indexer,
    assembler,
    scaler,
    lr
])

model = pipeline.fit(train_df)
print("Model berhasil dilatih.")


Model berhasil dilatih.


## Evaluasi Model

In [14]:
pred = model.transform(test_df)
pred.select("features", "prediction", "label").show(10, truncate=False)


+------------------------------------------------------------------------------------------------+----------+-----+
|features                                                                                        |prediction|label|
+------------------------------------------------------------------------------------------------+----------+-----+
|[1.3648298602175806,3.024621403350356,1.4818637838393365,1.2448006344226628,50.88284497358638]  |0.0       |0.0  |
|[1.3684029042782897,2.9497277300609435,0.45503331515274464,1.621971602850858,48.62142617733485] |0.0       |0.0  |
|[1.3721805218254548,3.5408603394129727,2.0034357093217925,0.09240982779624358,49.74500946827677]|0.0       |0.0  |
|[1.3792821769122579,3.34618186197351,0.4910192322726025,1.985856440501907,48.86730558688663]    |0.0       |0.0  |
|[1.3877790267328929,1.5151401393397432,3.4323079674623904,1.6777021241526024,49.80780080592754] |0.0       |0.0  |
|[1.3946055841977691,4.86003731223917,1.3542955065407436,1.3018703120229

In [15]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

acc = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(pred)
prec = MulticlassClassificationEvaluator(metricName="weightedPrecision").evaluate(pred)
rec = MulticlassClassificationEvaluator(metricName="weightedRecall").evaluate(pred)
f1 = MulticlassClassificationEvaluator(metricName="f1").evaluate(pred)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)


Accuracy : 0.9242761692650334
Precision: 0.9242761692650334
Recall   : 0.9242761692650334
F1 Score : 0.9242761692650334


## Prediksi 10 Data Pertama

In [16]:
pred.select(feature_cols + ["prediction", "label"]).show(10)


+------------------+------------------+------------------+------------------+------------------+----------+-----+
|       Temperature|          Humidity|        Wind_Speed|       Cloud_Cover|          Pressure|prediction|label|
+------------------+------------------+------------------+------------------+------------------+----------+-----+
| 10.01364912242489|  60.1791516406965| 8.617368361421986| 36.12358113412887|1030.4828084246208|       0.0|  0.0|
|10.039864265107365| 58.68903531842889|  2.64612020089382| 47.06892105659205| 984.6844024314966|       0.0|  0.0|
|10.067580347337568|  70.4504606983298|11.650425419617669|  2.68169361396311|1007.4392870247194|       0.0|  0.0|
|10.119684631029305| 66.57705505422769| 2.855386333872638| 57.62870315572597| 989.6639688181684|       0.0|  0.0|
|10.182025348528125| 30.14587151345059|19.959636241892117| 48.68619690962419|1008.7109414300636|       0.0|  0.0|
|10.232111262648177| 96.69736584840452|  7.87553038096616| 37.77971872924477|1020.920264